# TP C4 — v0.3 : `config.json` et `report.json`

Mar 13/10, 08:30–09:20. Correction à 09:20, TP fini ou non. Référence v0.3 publiée à 09:30.

## Objectif

`fleetcheck` charge sa configuration depuis `config.json` et écrit son résultat dans `report.json`, lisible par `jq` ou un autre script. Une config absente ou mal formée arrête le script avec un message d'une ligne et le code `1`, sans trace Python.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et à vérifier chaque étape.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run fleetcheck.py` exécute le fichier tel qu'il est enregistré, puis laisse ses variables et ses fonctions dans le notebook. `run()` lance le script comme dans un terminal et affiche sa sortie standard, sa sortie d'erreur (en rouge) et son code de sortie. Enregistrez le fichier avant de relancer une vérification.
- Les vérifications qui cassent `config.json` travaillent sur une copie et remettent le fichier d'origine en place : ne les modifiez pas.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py`, `config.json` et de ce notebook. `report.json` est produit à chaque exécution : ne le commitez pas.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en C9 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    return r

## Point de départ

Votre `fleetcheck.py` de fin de C3 (v0.2). En retard : copiez `reference/fleetcheck.py` du dépôt public, à la version `v0.2`.

Le résumé texte de v0.2 ne change pas : `print` reste sur l'écran, le rapport s'ajoute. Le bloc principal reste au niveau du module, sans `if __name__ == "__main__":` (vu en C5).

## 1. Créer `config.json` — fin 08:35

À côté de `fleetcheck.py`, créez `config.json` (lanceur de JupyterLab, *Text File*, puis renommer) avec ce contenu :

```text
{
  "timeout_s": 2,
  "attempts": 3,
  "backoff_s": 0.5,
  "degraded_s": 1.0,
  "ping_timeout_s": 1,
  "drift_envs": ["prod"],
  "releases_url": "http://127.0.0.1:8000/api/v4/projects/fleetcheck/releases",
  "reports_dir": "reports",
  "archive_after_days": 7
}
```

Les clés serviront de C8 à C11. En v0.3, on vérifie seulement que le fichier se charge.

Vérification : le fichier se lit avec `json.load`, sans erreur, et contient les 9 clés.

In [ ]:
import json

with open("config.json", encoding="utf-8") as f:
    config = json.load(f)
check(list(config), ["timeout_s", "attempts", "backoff_s", "degraded_s", "ping_timeout_s",
                     "drift_envs", "releases_url", "reports_dir", "archive_after_days"])

## 2. `load_config(path)` — fin 08:50

Écrivez `load_config(path)` :

- elle renvoie le dict lu dans le fichier ;
- si le fichier manque ou si le JSON est faux, elle affiche avec `print` une ligne `config illisible (<path>) : <détail de l'erreur>`, puis arrête le script avec `sys.exit(1)`.

Appelez-la au début du script avec `"config.json"`.

In [ ]:
%run fleetcheck.py


Vérification avec un fichier présent : `load_config` renvoie le dict.

In [ ]:
%run fleetcheck.py
check(load_config("config.json")["attempts"], 3)

Vérification sans `config.json` : la cellule renomme le fichier en `config.bak`, lance le script, puis remet le nom. Sortie attendue, sans trace Python, et code de sortie `1` :

```text
config illisible (config.json) : [Errno 2] No such file or directory: 'config.json'
```

In [ ]:
from pathlib import Path

Path("config.json").rename("config.bak")
try:
    r = run()
finally:
    Path("config.bak").rename("config.json")
check(r.stdout, "config illisible (config.json) : [Errno 2] No such file or directory: 'config.json'\n")
check((r.stderr, r.returncode), ("", 1))

Si `load_config` ne marche toujours pas à 08:50 : chargez la config avec `json.load` sans gestion d'erreur et passez à l'étape 3. Le rapport compte plus que le message d'erreur.

## 3. Écrire `report.json` — fin 09:10

À chaque exécution, le script écrit `report.json` à la racine, au format suivant (extrait : 1 service et 1 invalide montrés) :

```text
{
  "generated_at": "2026-10-13T08:52:10+02:00",
  "summary": {"total": 9, "invalid": 2},
  "services": [{"name": "web-1", "env": "prod", "ip": "127.0.0.1", "port": 8001, "role": "web"}],
  "invalid_lines": [{"name": "api-3", "error": "port non entier : 'abc'"}]
}
```

- `services` : les serveurs valides, tels que `load` les renvoie (port en entier).
- `invalid_lines` : les invalides collectés par `load`.
- `summary` : `total` est le nombre de serveurs valides, `invalid` le nombre d'invalides.
- `generated_at` s'obtient avec cette ligne, après `from datetime import datetime` :

```text
datetime.now().astimezone().isoformat(timespec="seconds")
```

Le fichier doit rester lisible : indentation de 2 espaces, accents écrits tels quels.

Essayez ici, puis ouvrez `report.json` dans JupyterLab :

In [ ]:
r = run()


Vérification : le rapport se relit avec `json.load` ; clés, résumé, premier service, premier invalide, date avec fuseau horaire et indentation de 2 espaces.

In [ ]:
from datetime import datetime

r = run()
with open("report.json", encoding="utf-8") as f:
    report = json.load(f)
check(list(report), ["generated_at", "summary", "services", "invalid_lines"])
check(report["summary"], {"total": 9, "invalid": 2})
check(report["services"][0], {"name": "web-1", "env": "prod", "ip": "127.0.0.1", "port": 8001, "role": "web"})
check(report["invalid_lines"][0], {"name": "api-3", "error": "port non entier : 'abc'"})
check(datetime.fromisoformat(report["generated_at"]).tzinfo is not None, True)
check(Path("report.json").read_text(encoding="utf-8").startswith('{\n  "generated_at": '), True)

## 4. Config cassée — fin 09:20

La cellule ci-dessous copie `config.json` en `config.bak`, ajoute une virgule après la dernière valeur (`"archive_after_days": 7,`), lance le script, puis remet le fichier d'origine.

Sortie attendue : une seule ligne, sans trace Python, et le code de sortie `1`. Avec Python 3.14, le détail est :

```text
config illisible (config.json) : Illegal trailing comma before end of object: line 10 column 26 (char 259)
```

La position (`line`, `column`, `char`) dépend de la mise en forme de votre fichier ; seul le début de la ligne est vérifié.

In [ ]:
import shutil

shutil.copy("config.json", "config.bak")
texte = Path("config.json").read_text(encoding="utf-8")
Path("config.json").write_text(texte.rstrip().removesuffix("}").rstrip() + ",\n}\n", encoding="utf-8")
try:
    r = run()
finally:
    shutil.move("config.bak", "config.json")
check(r.stdout.startswith("config illisible (config.json) : Illegal trailing comma before end of object"), True)
check((len(r.stdout.splitlines()), r.stderr, r.returncode), (1, "", 1))

Config d'origine en place, `run()` redonne la sortie de v0.2. C'est le critère de fin du TP.

In [ ]:
r = run()
check(r.stdout, """9 serveurs valides, 2 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
invalide api-3 : port non entier : 'abc'
invalide web-3 : IP invalide : '300.0.0.1'
""")
check(r.returncode, 0)

## À éviter

- `json.dump` hors d'un `with` : fichier vide si le script plante avant la fermeture.
- Lancer le script depuis un autre dossier : `config.json` est cherché dans le dossier courant, pas à côté du script. Notez-le, la réponse vient en C13.
- `except Exception` autour du chargement : il cache aussi une faute de frappe dans votre code.

## Pour aller plus loin

Non corrigé en salle. Écrivez vos propres vérifications dans la cellule ci-dessous.

- Refuser une config à laquelle manque une clé attendue.
- Écrire le rapport dans `Path(config["reports_dir"]) / "report.json"`, en créant le dossier s'il n'existe pas.

In [ ]:
%run fleetcheck.py


## En retard

La version de référence v0.3 (`fleetcheck.py` et `config.json`) est publiée dans `reference/` à 09:30. Copiez-la dans votre dépôt pour démarrer le TP C5.